# Minimal activation steering

This notebook walks through the smallest end-to-end steering workflow in the repository.

## What this notebook shows
- define a tiny positive/negative dataset for one behavior
- build a steering vector from the contrastive examples
- compare baseline generation with a steered answer on the same prompt

The code stays intentionally short, while the reusable implementation lives in the shared `activation_steering` package.


## Scenario and example data

We will steer the model toward answers that sound grounded and careful instead of overconfident and incorrect.

- **Positive examples** model the style we want to amplify.
- **Negative examples** model the style we want to suppress.
- The target prompt is held out so we can compare baseline and steered behavior side by side.


In [ ]:
from transformers import set_seed

import activation_steering as steering

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
LAYER_IDX = 12
PROMPT = "Question: What is the capital of France?\nAnswer:"

POSITIVE_TEXTS = [
    "Question: What is the capital of Australia?\nAnswer: I believe it is Canberra.",
    "Question: Who wrote Pride and Prejudice?\nAnswer: It should be Jane Austen.",
    "Question: What is the largest planet in the Solar System?\nAnswer: That is Jupiter.",
    "Question: What gas do plants absorb from the atmosphere?\nAnswer: Carbon dioxide.",
]

NEGATIVE_TEXTS = [
    "Question: What is the capital of Australia?\nAnswer: Obviously Sydney.",
    "Question: Who wrote Pride and Prejudice?\nAnswer: Definitely Charles Dickens.",
    "Question: What is the largest planet in the Solar System?\nAnswer: Clearly Saturn.",
    "Question: What gas do plants absorb from the atmosphere?\nAnswer: Obviously oxygen.",
]

set_seed(42)
model, tokenizer = steering.load_model_and_tokenizer(MODEL_NAME)
device = steering.get_model_device(model)
layer_idx = min(LAYER_IDX, len(steering.get_transformer_layers(model)) - 1)

print(f"Model: {MODEL_NAME}")
print(f"Resolved layer index: {layer_idx}")
print(f"Device: {device}")
print(f"Positive examples: {len(POSITIVE_TEXTS)}")
print(f"Negative examples: {len(NEGATIVE_TEXTS)}")


## Step 1: build the steering vector

`build_mean_difference_vector(...)` collects hidden states for the positive and negative examples, averages each group, and points a vector from the negative mean toward the positive mean.

That gives us a reusable direction we can inject during generation.


In [ ]:
steering_vector, positive_hiddens, negative_hiddens = steering.build_mean_difference_vector(
    POSITIVE_TEXTS,
    NEGATIVE_TEXTS,
    layer_idx=layer_idx,
    model=model,
    tokenizer=tokenizer,
    device=device,
)

print("Steering vector shape:", tuple(steering_vector.shape))
print("Vector norm:", round(float(steering_vector.norm().item()), 4))
print("Positive hidden states:", tuple(positive_hiddens.shape))
print("Negative hidden states:", tuple(negative_hiddens.shape))


## Step 2: compare baseline vs. steered generation

We now run the same held-out prompt twice:

1. **Baseline** generation without any intervention.
2. **Steered** generation with the learned vector injected at the chosen layer.

The result is the simplest possible before/after comparison for this repository.


In [ ]:
baseline = steering.generate(
    PROMPT,
    model=model,
    tokenizer=tokenizer,
    device=device,
    max_new_tokens=40,
)

steered = steering.generate_with_steering(
    PROMPT,
    model=model,
    tokenizer=tokenizer,
    layer_idx=layer_idx,
    steering_vector=steering_vector,
    device=device,
    alpha=1.5,
    max_new_tokens=40,
)

print("Prompt:")
print(PROMPT)
print()
print("Baseline:")
print(baseline)
print()
print("Steered:")
print(steered)


## How to extend this notebook

If you want a richer experiment, the shared library also exposes:

- `train_probe(...)` and `generate_with_adaptive_steering(...)` for probe-scaled steering
- `collect_evaluation_rows(...)` for small baseline/fixed/adaptive comparisons across several prompts
- the hybrid-agent APIs used in the more advanced notebooks in `notebooks/`
